# Safety Cases and Goal Structuring Notation (GSN)

**Module 08 · Standards, Regulation and Safety Case** · Notebook 4 of 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/08_Standards_Regulation_and_Safety_Case/04_safety_case_gsn.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Advanced | ~80 min | [01 Standards integration](01_standards_integration.ipynb); recommended: [HARA](../03_Functional_Safety_ISO26262/02_hara.ipynb), [FTA](../03_Functional_Safety_ISO26262/04_fault_tree_analysis.ipynb) | GSN Community Standard v3 (2021), UL 4600 (3rd ed., 2023), ISO/TS 5083:2025, ISO/PAS 8800:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Explain what a **safety case** is in terms of **claims, arguments and evidence**, and why automated driving standards increasingly centre on it.
2. Name and draw the **GSN elements** of the GSN Community Standard (Version 3, 2021) and the two relationships *SupportedBy* and *InContextOf*.
3. Describe **argument patterns**, **modules** and **confidence arguments**, including the idea of **defeaters** in eliminative argumentation.
4. Summarise what **UL 4600**, **ISO/TS 5083:2025**, **ISO/PAS 8800:2024** and **AMLAS** expect of an assurance argument.
5. Build, validate and render a GSN argument in Python for *"the AEB pedestrian function is acceptably safe within its ODD"*, combining ISO 26262, ISO 21448, ISO/PAS 8800 and ISO/SAE 21434 evidence.
6. Automate routine checks: undeveloped goals, type errors, stale evidence and open defeaters.

## 1. Setup

In [ ]:
import textwrap
from dataclasses import dataclass, field
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, Circle, Ellipse, FancyBboxPatch, FancyArrowPatch
from IPython.display import display, Markdown

rng = np.random.default_rng(42)   # no randomness is used below; seeded for reproducibility by convention
TODAY = date(2026, 10, 28)        # fixed "assessment date" so that the freshness checks are reproducible

OI = {"orange": "#E69F00", "sky": "#56B4E9", "green": "#009E73", "yellow": "#F0E442",
      "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7", "grey": "#999999"}
print("Setup complete. Assessment date:", TODAY.isoformat())

## 2. What is a safety case?

A **safety case** is a structured argument, supported by evidence, that a system is acceptably safe for a specific application in a specific operating environment. It has three ingredients:

| Ingredient | Question it answers | AEB example |
|---|---|---|
| **Claim** | What do we assert? | *"Unintended full braking is avoided."* |
| **Argument** | Why does the evidence support the claim? | *"The erroneous request needs two independent faults; common causes are excluded by the DFA."* |
| **Evidence** | What facts do we have? | Fault tree and minimal cut sets, FMEDA metrics, test reports |

Evidence alone is not a safety case: a folder of test reports does not explain *why* those tests are sufficient. Conversely, an argument without evidence is only an assertion. The argument is the part that is most often left implicit — and the part that reviewers, assessors and regulators most need to see.

Why it matters for automated driving: there is no single prescriptive standard that, if followed, guarantees the safety of an ML-based driving function. Standards such as UL 4600 therefore ask the developer to **argue** safety, and assess the argument (Section 4).

## 3. Goal Structuring Notation (GSN)

GSN is a graphical notation for arguments, maintained by the Safety-Critical Systems Club's Assurance Case Working Group. The current version is the **GSN Community Standard, Version 3** (SCSC-141C, May 2021). Its core elements:

| Element | Shape | Role |
|---|---|---|
| **Goal** | rectangle | a claim (a proposition that is true or false) |
| **Strategy** | parallelogram | the reasoning step that breaks a goal into sub-goals |
| **Solution** | circle | a reference to an item of evidence |
| **Context** | rounded rectangle | scope, definitions, the ODD, referenced artefacts |
| **Assumption** | ellipse marked **A** | a statement taken as true without further support |
| **Justification** | ellipse marked **J** | why a goal or strategy is acceptable |
| **Away goal** | rectangle with a module band | a goal supported in another module of the argument |
| **Undeveloped** | hollow diamond below an element | the element still needs to be developed |

Two relationships connect them:

* **SupportedBy** (solid arrowhead) — inferential or evidential support: Goal → Goal/Strategy/Solution, Strategy → Goal.
* **InContextOf** (hollow arrowhead) — contextual links: Goal/Strategy → Context/Assumption/Justification.

**Patterns** are reusable argument templates with placeholders, *multiplicity* ("one sub-goal per hazard") and *optionality*. **Modules** split a large argument into parts (e.g. a safety case and a cybersecurity case) that refer to each other through away goals and contracts. Version 3 also introduces **assurance claim points** and **dialectic** extensions, which let an argument record challenges (counter-arguments) — the subject of Section 9.

## 4. Standards that expect an assurance argument

**UL 4600 (3rd ed., 2023)** — *Standard for Safety for the Evaluation of Autonomous Products.* UL 4600 is **safety-case-centric**: rather than prescribing a development process, it states what a safety case for an autonomous product must cover and how it is assessed. The third edition (March 2023) added autonomous trucking. UL 4600 requires **safety performance indicators (SPIs)**: metrics, supported by evidence, that use a threshold comparison to condition a claim in the safety case — so that field data can show when a design-time claim no longer holds. Relying on lagging indicators (counts of mishaps) alone is not considered sufficient; leading indicators are required too.

**ISO/TS 5083:2025** — *Road vehicles — Safety for automated driving systems — Design, verification and validation* (first edition, April 2025). Guidance on achieving and demonstrating the safety of SAE Level 3 and 4 automated driving systems, covering safety by design, verification and validation, post-deployment activities and some cybersecurity considerations. It includes guidance on describing the ADS **safety case**, including a layered safety-case structure.

**ISO/PAS 8800:2024** — *Road vehicles — Safety and artificial intelligence.* Intended to be used together with ISO 26262 and ISO 21448. Its clause 8, **"Assurance arguments for AI systems"**, covers AI-specific considerations in assurance arguments, how to structure them, and how to evaluate them.

**AMLAS** — *Guidance on the Assurance of Machine Learning in Autonomous Systems* (Hawkins et al., University of York, 2021). A process with six stages — ML safety assurance scoping, ML safety requirements, data management, model learning, model verification and model deployment — each with a GSN **safety case pattern** for the corresponding part of the ML argument. Data requirements, for example, are argued along relevance, completeness, accuracy and balance.

> **Scope note:** the descriptions above are limited to what the publicly available scope statements, tables of contents and papers state. For normative detail, consult the standards themselves.

## 5. A tiny GSN library

The library below has three parts: a typed node model, a graph with the two GSN relationships and their typing rules, and a validator. Rendering follows in Section 6.

In [ ]:
NODE_TYPES = ("Goal", "Strategy", "Solution", "Context", "Assumption", "Justification", "AwayGoal")

# Permitted relationships (GSN Community Standard v3, core notation)
SUPPORTED_BY = {"Goal": {"Goal", "Strategy", "Solution", "AwayGoal"},
                "Strategy": {"Goal", "AwayGoal"}}
IN_CONTEXT_OF = {"Goal": {"Context", "Assumption", "Justification"},
                 "Strategy": {"Context", "Assumption", "Justification"},
                 "AwayGoal": {"Context", "Assumption", "Justification"}}
PREFIX = {"Goal": "G", "Strategy": "S", "Solution": "Sn", "Context": "C",
          "Assumption": "A", "Justification": "J", "AwayGoal": "G"}

@dataclass
class Node:
    id: str
    type: str
    text: str
    undeveloped: bool = False
    meta: dict = field(default_factory=dict)   # e.g. artefact, link, date, version, module

class Argument:
    def __init__(self, name):
        self.name, self.nodes, self.edges = name, {}, []   # edges: (src, dst, rel)

    def add(self, id, type, text, undeveloped=False, **meta):
        if type not in NODE_TYPES:
            raise ValueError(f"unknown GSN element type {type!r}")
        if id in self.nodes:
            raise ValueError(f"duplicate id {id}")
        self.nodes[id] = Node(id, type, text, undeveloped, meta)
        return id

    def supported_by(self, parent, *children):
        for c in children:
            self.edges.append((parent, c, "SupportedBy"))

    def in_context_of(self, node, *ctx):
        for c in ctx:
            self.edges.append((node, c, "InContextOf"))

    # convenience queries
    def children(self, n, rel="SupportedBy"):
        return [d for s, d, r in self.edges if s == n and r == rel]

    def parents(self, n, rel="SupportedBy"):
        return [s for s, d, r in self.edges if d == n and r == rel]

    def roots(self):
        return [n for n, nd in self.nodes.items()
                if nd.type in ("Goal", "Strategy") and not self.parents(n)]

    def of_type(self, t):
        return [n for n, nd in self.nodes.items() if nd.type == t]

print("GSN library ready:", ", ".join(NODE_TYPES))

### 5.1 Validation rules

Some rules follow directly from the notation (type rules), others are good practice that a reviewer would check by hand:

| Rule | Severity | Check |
|---|---|---|
| R1 | error | *SupportedBy* only between permitted types (e.g. a strategy may not point straight at a solution) |
| R2 | error | *InContextOf* only to context, assumption or justification |
| R3 | error | every leaf goal is supported, or explicitly marked **undeveloped** |
| R4 | warning | every undeveloped element is listed as an open item |
| R5 | error | every strategy has at least one sub-goal (or is marked undeveloped) |
| R6 | error | exactly one root, and it is a goal |
| R7 | error | the *SupportedBy* graph is acyclic |
| R8 | warning | solutions, contexts, assumptions and justifications are connected to something |
| R9 | error | every solution references an evidence artefact |
| R10 | info | goals are phrased as propositions, not questions or instructions |

In [ ]:
IMPERATIVE = ("ensure", "verify", "check", "perform", "test", "show", "demonstrate", "analyse", "analyze")

def validate(arg):
    f = []
    def add(rule, sev, node, msg):
        f.append({"rule": rule, "severity": sev, "node": node, "message": msg})
    for s, d, r in arg.edges:
        for x in (s, d):
            if x not in arg.nodes:
                add("R0", "error", x, f"edge {s}->{d} refers to unknown node")
        if s not in arg.nodes or d not in arg.nodes:
            continue
        ts, td = arg.nodes[s].type, arg.nodes[d].type
        allowed = (SUPPORTED_BY if r == "SupportedBy" else IN_CONTEXT_OF).get(ts, set())
        if td not in allowed:
            add("R1" if r == "SupportedBy" else "R2", "error", s, f"{ts} {s} --{r}--> {td} {d} is not permitted")
    for n, nd in arg.nodes.items():
        kids = arg.children(n)
        if nd.type == "Goal" and not kids and not nd.undeveloped:
            add("R3", "error", n, "leaf goal is neither supported nor marked undeveloped")
        if nd.type == "Strategy" and not kids and not nd.undeveloped:
            add("R5", "error", n, "strategy has no sub-goals")
        if nd.undeveloped:
            add("R4", "warning", n, f"undeveloped {nd.type.lower()}: open item")
        if nd.type in ("Solution", "Context", "Assumption", "Justification") and not (
                arg.parents(n) or arg.parents(n, "InContextOf")):
            add("R8", "warning", n, f"{nd.type.lower()} is not connected")
        if nd.type == "Solution" and not nd.meta.get("artefact"):
            add("R9", "error", n, "solution does not reference an evidence artefact")
        if nd.type == "Goal":
            t = nd.text.strip().lower()
            if t.endswith("?") or t.split(" ")[0] in IMPERATIVE:
                add("R10", "info", n, "goal should be a proposition (true/false), not a question or instruction")
    roots = arg.roots()
    if len(roots) != 1 or arg.nodes[roots[0]].type != "Goal":
        add("R6", "error", ",".join(roots) or "-", f"expected one goal as root, found {roots}")
    # R7: cycle detection on SupportedBy (depth-first search with colours)
    state = {}
    def dfs(n):
        state[n] = 1
        for c in arg.children(n):
            if c in arg.nodes and state.get(c) == 1:
                add("R7", "error", n, f"cycle via {n}->{c}"); continue
            if c in arg.nodes and c not in state:
                dfs(c)
        state[n] = 2
    for n in arg.nodes:
        if n not in state:
            dfs(n)
    order = {"error": 0, "warning": 1, "info": 2}
    return (pd.DataFrame(f, columns=["rule", "severity", "node", "message"])
              .sort_values(["severity", "rule"], key=lambda s: s.map(order) if s.name == "severity" else s)
              .reset_index(drop=True))

def summarise(findings, name):
    counts = findings["severity"].value_counts().reindex(["error", "warning", "info"], fill_value=0)
    verdict = "well-formed" if counts["error"] == 0 else "NOT well-formed"
    print(f"{name}: {counts['error']} error(s), {counts['warning']} warning(s), {counts['info']} info -> {verdict}")

### 5.2 Try the validator on a deliberately flawed argument

In [ ]:
bad = Argument("flawed example")
bad.add("G1", "Goal", "The AEB function is safe")
bad.add("S1", "Strategy", "Argument over test campaigns")
bad.add("Sn1", "Solution", "Track test report")                 # no artefact reference
bad.add("G2", "Goal", "Ensure the detector works at night")       # instruction, and unsupported
bad.add("C1", "Context", "ODD definition")
bad.add("G3", "Goal", "Is the braking distance sufficient?")      # question
bad.supported_by("G1", "S1", "G2", "G3")
bad.supported_by("S1", "Sn1")                                     # strategy -> solution: type error
bad.supported_by("C1", "G3")                                      # context cannot support anything
bad.in_context_of("G1", "Sn1")                                    # InContextOf to a solution: type error

bad_findings = validate(bad)
display(bad_findings)
summarise(bad_findings, bad.name)

## 6. Rendering GSN diagrams

The renderer uses the standard shapes. Layout: a tidy tree over *SupportedBy*, with context-type elements placed to the right of the element they belong to. Each subtree reserves enough width for its own context elements, so nothing overlaps.

In [ ]:
BOX_W, BOX_H, CTX_W, CTX_H, GAP, DY = 3.3, 1.55, 3.0, 1.4, 0.4, 2.7
FILL = {"Goal": "#DCEBF7", "Strategy": "#FFF3C4", "Solution": "#D5F0E5", "Context": "#EFEFEF",
        "Assumption": "#F6E1EC", "Justification": "#F6E1EC", "AwayGoal": "#DCEBF7"}

def _wrap(t, width):
    return "\n".join(textwrap.wrap(t, width))

def draw_node(ax, nd, x, y):
    t = nd.type
    lab = f"{nd.id}\n" + _wrap(nd.text, 30 if t not in ("Solution",) else 15)
    fs = 7.2
    if t == "Goal":
        ax.add_patch(Rectangle((x - BOX_W / 2, y - BOX_H / 2), BOX_W, BOX_H, fc=FILL[t], ec="black", lw=1.1))
    elif t == "AwayGoal":
        ax.add_patch(Rectangle((x - BOX_W / 2, y - BOX_H / 2), BOX_W, BOX_H, fc=FILL[t], ec="black", lw=1.1))
        ax.plot([x - BOX_W / 2, x + BOX_W / 2], [y - BOX_H / 2 + 0.32] * 2, color="black", lw=1)
        ax.add_patch(Rectangle((x - BOX_W / 2 + 0.08, y - BOX_H / 2 + 0.07), 0.3, 0.16, fc="white", ec="black", lw=0.8))
        ax.text(x + 0.1, y - BOX_H / 2 + 0.15, nd.meta.get("module", "module"), ha="center", va="center", fontsize=6, style="italic")
        y = y + 0.16
        lab = f"{nd.id}\n" + _wrap(nd.text, 30)
    elif t == "Strategy":
        s = 0.25
        ax.add_patch(Polygon([(x - BOX_W / 2 + s, y + BOX_H / 2), (x + BOX_W / 2 + s, y + BOX_H / 2),
                              (x + BOX_W / 2 - s, y - BOX_H / 2), (x - BOX_W / 2 - s, y - BOX_H / 2)],
                             fc=FILL[t], ec="black", lw=1.1))
    elif t == "Solution":
        ax.add_patch(Circle((x, y), 0.9, fc=FILL[t], ec="black", lw=1.1))
        fs = 6.6
    elif t == "Context":
        ax.add_patch(FancyBboxPatch((x - CTX_W / 2 + 0.15, y - CTX_H / 2 + 0.1), CTX_W - 0.3, CTX_H - 0.2,
                                    boxstyle="round,pad=0.1,rounding_size=0.3", fc=FILL[t], ec="black", lw=1.0))
        lab = f"{nd.id}\n" + _wrap(nd.text, 28)
    else:  # Assumption / Justification
        ax.add_patch(Ellipse((x, y), CTX_W, CTX_H + 0.15, fc=FILL[t], ec="black", lw=1.0))
        ax.text(x + 0.37 * CTX_W, y - 0.37 * (CTX_H + 0.15), t[0], fontsize=9, fontweight="bold", ha="left", va="top")
        lab = f"{nd.id}\n" + _wrap(nd.text, 25)
    ax.text(x, y, lab, ha="center", va="center", fontsize=fs)
    if nd.undeveloped:
        yb = y - (0.9 if t == "Solution" else BOX_H / 2)
        ax.add_patch(Polygon([(x, yb), (x + 0.18, yb - 0.18), (x, yb - 0.36), (x - 0.18, yb - 0.18)],
                             fc="white", ec="black", lw=1.0))

def render(arg, root=None, max_depth=None, title=None, highlight=()):
    root = root or arg.roots()[0]
    ctx_of = lambda n: arg.children(n, "InContextOf")
    placed = set()
    def kids(n, depth):
        if max_depth is not None and depth >= max_depth:
            return []
        out = []
        for c in arg.children(n):
            if c not in placed and c in arg.nodes:
                placed.add(c); out.append(c)
        return out
    # build a tree (first parent wins for shared nodes)
    tree, stack = {}, [(root, 0)]
    placed.add(root)
    while stack:
        n, d = stack.pop(0)
        tree[n] = (kids(n, d), d)
        stack.extend((c, d + 1) for c in tree[n][0])
    width = {}
    def w(n):
        own = BOX_W + len(ctx_of(n)) * (CTX_W + GAP)
        ch = tree[n][0]
        kw = sum(w(c) for c in ch) + GAP * max(len(ch) - 1, 0)
        width[n] = max(own, kw)
        return width[n]
    w(root)
    pos = {}
    def place(n, left):
        W, ch = width[n], tree[n][0]
        own = BOX_W + len(ctx_of(n)) * (CTX_W + GAP)
        if ch:
            kw = sum(width[c] for c in ch) + GAP * (len(ch) - 1)
            cur = left + (W - kw) / 2
            for c in ch:
                place(c, cur); cur += width[c] + GAP
            x = np.mean([pos[c][0] for c in ch])
        else:
            x = left + W / 2 - (own - BOX_W) / 2
        x = float(np.clip(x, left + BOX_W / 2, left + W - own + BOX_W / 2))
        pos[n] = (x, -tree[n][1] * DY)
    place(root, 0.0)

    depth_max = max(d for _, d in tree.values())
    fig, ax = plt.subplots(figsize=(max(8, width[root] * 0.72), 1.4 + 1.95 * (depth_max + 1)))
    xmax = [max(x for x, _ in pos.values()) + BOX_W / 2 + 0.4]
    for n, (x, y) in pos.items():                                 # SupportedBy edges
        for c in tree[n][0]:
            cx, cy = pos[c]
            top = cy + (0.9 if arg.nodes[c].type == "Solution" else BOX_H / 2)
            ax.add_patch(FancyArrowPatch((x, y - BOX_H / 2), (cx, top), arrowstyle="-|>", mutation_scale=11,
                                         color="black", lw=1.0, shrinkA=0, shrinkB=0))
    for n, (x, y) in pos.items():                                 # context elements and InContextOf edges
        for i, c in enumerate(ctx_of(n)):
            cx = x + BOX_W / 2 + GAP + CTX_W / 2 + i * (CTX_W + GAP)
            draw_node(ax, arg.nodes[c], cx, y)
            xmax.append(cx + CTX_W / 2 + 0.3)
            x0 = x + BOX_W / 2 + (0.25 if arg.nodes[n].type == "Strategy" else 0)
            if i == 0:   # nearest context: straight arrow
                start, end, style = (x0, y), (cx - CTX_W / 2, y), "arc3,rad=0"
            else:        # further contexts: arc over the nearer ones
                start, end, style = (x0 - 0.3, y + BOX_H / 2), (cx - 0.5, y + CTX_H / 2), "arc3,rad=-0.12"
            ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=11, connectionstyle=style,
                                         fc="white", ec="black", lw=1.0, shrinkA=0, shrinkB=0))
        draw_node(ax, arg.nodes[n], x, y)
        if n in highlight:
            ax.add_patch(Rectangle((x - BOX_W / 2 - 0.1, y - BOX_H / 2 - 0.1), BOX_W + 0.2, BOX_H + 0.2,
                                   fill=False, ec=OI["vermillion"], lw=2.2, ls="--"))
        if max_depth is not None and tree[n][1] >= max_depth and arg.children(n):
            ax.text(x, y - BOX_H / 2 - 0.15, "▼ continued", ha="center", va="top", fontsize=6, color=OI["blue"])
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - BOX_W / 2 - 0.3, max(xmax))
    ax.set_ylim(min(ys) - 1.3, max(ys) + 1.6)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title or arg.name, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()
    return pos

# A legend of the GSN shapes
legend = Argument("GSN element legend")
for i, (t, txt) in enumerate([("Goal", "claim"), ("Strategy", "reasoning step"), ("Solution", "evidence"),
                              ("Context", "scope / definition"), ("Assumption", "taken as true"),
                              ("Justification", "why acceptable"), ("AwayGoal", "goal in another module")]):
    legend.add(f"{PREFIX[t]}{i}", t, f"{t}: {txt}", module="Module X")
legend.nodes["G0"].undeveloped = True
fig, ax = plt.subplots(figsize=(15, 2.6))
for i, nd in enumerate(legend.nodes.values()):
    draw_node(ax, nd, 1.8 + i * 3.6, 0)
ax.add_patch(FancyArrowPatch((0.5, -1.55), (2.3, -1.55), arrowstyle="-|>", mutation_scale=12, color="black"))
ax.text(2.5, -1.55, "SupportedBy", va="center", fontsize=8)
ax.add_patch(FancyArrowPatch((5.0, -1.55), (6.8, -1.55), arrowstyle="-|>", mutation_scale=12, fc="white", ec="black"))
ax.text(7.0, -1.55, "InContextOf", va="center", fontsize=8)
ax.text(9.6, -1.55, "◇ below an element = undeveloped (shown on G0)", va="center", fontsize=8)
ax.set_xlim(0, 25.6); ax.set_ylim(-2.0, 1.1); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("GSN elements (GSN Community Standard v3)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()
print("Text summary: goal = rectangle, strategy = parallelogram, solution = circle, context = rounded rectangle, "
      "assumption and justification = ellipses marked A and J, away goal = rectangle with a module band; "
      "SupportedBy uses a filled arrowhead, InContextOf a hollow one.")

## 7. The AEB pedestrian argument

The top claim is *"The AEB pedestrian function is acceptably safe within its defined ODD"*. The first strategy argues over **four complementary sources of hazardous behaviour**, each with its own standard:

| Branch | Source of hazardous behaviour | Standard | Course evidence |
|---|---|---|---|
| G2 | E/E malfunctions (random hardware and systematic faults) | ISO 26262:2018 | Module 03 |
| G3 | Functional insufficiencies and triggering conditions | ISO 21448:2022 | Module 04 |
| G4 | ML-specific insufficiencies (data, uncertainty, distribution shift) | ISO/PAS 8800:2024 | Modules 01, 06 |
| G5 | Cybersecurity threats with safety impact | ISO/SAE 21434:2021 | Modules 02, 05 |

**About branch G5.** Safety and security arguments are usually kept in separate documents — a safety case and a cybersecurity case — although an attack on an AI-based perception component can directly cause a safety-goal violation. This branch is motivated by the author's paper *"Integrating Cybersecurity Into the AI Safety Assurance Argument: A GSN Pattern for AI-Based Perception Components in Highly Automated Driving"* (Patel & Jung, SAFECOMP 2026 Workshops, LNCS, 2026), which, as its title states, proposes a GSN pattern that brings cybersecurity into the AI safety assurance argument for perception components. The branch below is a simplified teaching illustration of that idea — security claims that feed the safety argument, plus an away goal to the cybersecurity case — and **not** a reproduction of the published pattern; consult the paper for the pattern itself.

### 7.1 Patterns: one sub-argument per safety goal

The ISO 26262 branch is generated by a **pattern** with multiplicity: for each safety goal from the HARA, instantiate a goal and its evidence. The same function would produce the branch for any number of safety goals.

> **Illustrative data:** dates, versions and artefact names below are synthetic, chosen for teaching; they are not records of a real project.

In [ ]:
FS = "../03_Functional_Safety_ISO26262/"; SO = "../04_SOTIF_ISO21448/"; CY = "../05_Cybersecurity_ISO21434/"
AI = "../06_AI_Safety_ISO_PAS_8800/"; PE = "../01_Perception/"; FR = "../02_Failures_and_Robustness/"
RT = "../07_Runtime_Assurance_and_Operations/"
DEPLOYED_MODEL = "det-v1.4"          # version of the pedestrian detector in the release under assessment

def safety_goal_pattern(arg, strategy, safety_goals):
    # GSN pattern with multiplicity: one goal + evidence per safety goal {SG}.
    for k, (sg, text, asil) in enumerate(safety_goals, start=1):
        g = arg.add(f"G2.{k}", "Goal", f"{sg} '{text}' ({asil}) is met")
        sn_a = arg.add(f"Sn2.{k}a", "Solution", f"FTA and DFA for {sg}",
                       artefact="Fault tree, minimal cut sets, beta-factor analysis",
                       link=FS + "04_fault_tree_analysis.ipynb", date="2026-09-15", version="fta-r3")
        sn_b = arg.add(f"Sn2.{k}b", "Solution", f"FMEDA metrics for {sg}",
                       artefact="SPFM / LFM / PMHF calculation", link=FS + "03_fmea_and_fmeda.ipynb",
                       date="2026-07-01", version="fmeda-r5")
        sn_c = arg.add(f"Sn2.{k}c", "Solution", f"Verification of {sg}",
                       artefact="Requirements-based and fault-injection tests",
                       link=FS + "05_verification_and_validation.ipynb", date="2025-06-20", version="vv-r2")
        arg.supported_by(strategy, g)
        arg.supported_by(g, sn_a, sn_b, sn_c)

aeb = Argument("AEB pedestrian safety argument")
a = aeb.add
a("G1", "Goal", "The AEB pedestrian function is acceptably safe within its defined ODD")
a("C1", "Context", "ODD: urban roads up to 60 km/h, day and lit night, dry or wet")
a("C2", "Context", "Item: AEB incl. brake request path (HARA, notebook 03/02)")
a("C3", "Context", "Acceptably safe: no unreasonable risk; quantitative validation target agreed")
a("S1", "Strategy", "Argue over the four sources of hazardous behaviour")
a("J1", "Justification", "E/E faults, functional and ML insufficiencies and attacks are complementary causes")
aeb.in_context_of("G1", "C1", "C2", "C3"); aeb.in_context_of("S1", "J1")
aeb.supported_by("G1", "S1")

# --- ISO 26262 branch (pattern instantiation) ---
a("G2", "Goal", "Unreasonable risk from E/E malfunctions is avoided (ISO 26262)")
a("S2", "Strategy", "Argue over each safety goal from the HARA")
a("C4", "Context", "Safety goals and ASILs from the HARA")
aeb.supported_by("S1", "G2"); aeb.supported_by("G2", "S2"); aeb.in_context_of("S2", "C4")
safety_goal_pattern(aeb, "S2", [("SG-01", "avoid unintended AEB deceleration", "ASIL D"),
                                ("SG-02", "avoid omission of AEB deceleration when a collision is imminent", "ASIL per HARA")])

# --- ISO 21448 branch ---
a("G3", "Goal", "Risk from functional insufficiencies is acceptably low (ISO 21448)")
a("S3", "Strategy", "Argue over known (Area 2) and unknown (Area 3) hazardous scenarios")
a("G3.1", "Goal", "Known triggering conditions are identified and their risk is reduced")
a("G3.2", "Goal", "Residual risk from unknown scenarios meets the validation target")
a("A1", "Assumption", "Simulation is representative of the ODD for the tested scenarios")
a("Sn3.1a", "Solution", "Triggering-condition analysis", artefact="Triggering-condition catalogue",
  link=SO + "02_triggering_conditions_and_scenarios.ipynb", date="2026-08-30", version="tc-r4")
a("Sn3.1b", "Solution", "Criticality metrics and RSS", artefact="TTC / RSS evaluation",
  link=SO + "05_criticality_metrics_and_rss.ipynb", date="2026-09-02", version="crit-r1")
a("Sn3.2a", "Solution", "Simulation-based validation", artefact="Scenario-based simulation campaign",
  link=SO + "04_simulation_based_validation.ipynb", date="2026-09-20", version="sim-r6", model="det-v1.4")
a("Sn3.2b", "Solution", "Statistical validation", artefact="Zero-failure distance and confidence bound",
  link=AI + "07_statistical_safety_validation.ipynb", date="2026-09-25", version="stat-r2", model="det-v1.4")
aeb.supported_by("S1", "G3"); aeb.supported_by("G3", "S3"); aeb.supported_by("S3", "G3.1", "G3.2")
aeb.supported_by("G3.1", "Sn3.1a", "Sn3.1b"); aeb.supported_by("G3.2", "Sn3.2a", "Sn3.2b")
aeb.in_context_of("G3.2", "A1")

# --- ISO/PAS 8800 branch ---
a("G4", "Goal", "ML-specific insufficiencies of the pedestrian detector are acceptably mitigated (ISO/PAS 8800)")
a("S4", "Strategy", "Argue over data, model uncertainty and runtime monitoring")
a("G4.1", "Goal", "Training and test data are sufficient for the ODD")
a("G4.2", "Goal", "Detection confidence is calibrated and its uncertainty is quantified")
a("G4.3", "Goal", "Out-of-distribution inputs are detected and lead to a safe reaction")
a("Sn4.1", "Solution", "Dataset coverage analysis", artefact="Dataset analysis report",
  link=PE + "08_driving_datasets.ipynb", date="2026-05-10", version="data-r3")
a("Sn4.2a", "Solution", "Calibration (ECE)", artefact="Reliability diagram, ECE, temperature scaling",
  link=AI + "04_calibration.ipynb", date="2026-06-12", version="cal-r2", model="det-v1.3")
a("Sn4.2b", "Solution", "Conformal prediction sets", artefact="Coverage test of conformal sets",
  link=AI + "05_conformal_prediction.ipynb", date="2026-09-28", version="cp-r1", model="det-v1.4")
a("Sn4.3", "Solution", "OOD monitor evaluation", artefact="OOD detector AUROC and fallback test",
  link=SO + "03_ood_monitoring_as_safety_measure.ipynb", date="2026-09-05", version="ood-r3", model="det-v1.4")
aeb.supported_by("S1", "G4"); aeb.supported_by("G4", "S4"); aeb.supported_by("S4", "G4.1", "G4.2", "G4.3")
aeb.supported_by("G4.1", "Sn4.1"); aeb.supported_by("G4.2", "Sn4.2a", "Sn4.2b"); aeb.supported_by("G4.3", "Sn4.3")

# --- ISO/SAE 21434 branch (motivated by Patel & Jung, 2026) ---
a("G5", "Goal", "Cybersecurity threats cannot lead to a violation of the safety goals (ISO/SAE 21434)")
a("C5", "Context", "Motivation: GSN pattern integrating security into the AI safety argument (Patel & Jung, 2026)")
a("S5", "Strategy", "Argue over threat scenarios affecting the AI-based perception and brake path")
a("A2", "Assumption", "Attacker capabilities are bounded as rated in the TARA")
a("G5.1", "Goal", "Threat scenarios with safety impact are identified and risk-rated")
a("G5.2", "Goal", "The detector is robust to adversarial perturbations within the threat model")
a("G5.3", "Goal", "Injected brake commands on the vehicle network are detected")
a("G5.4", "Goal", "Security claims remain valid over the vehicle lifetime", undeveloped=True)
a("G5.5", "AwayGoal", "Cybersecurity case: risks treated per ISO/SAE 21434", module="CS-case")
a("Sn5.1", "Solution", "TARA", artefact="TARA worksheet (attack potential, risk values)",
  link=CY + "02_tara.ipynb", date="2026-08-14", version="tara-r3")
a("Sn5.2", "Solution", "Adversarial robustness tests", artefact="FGSM/PGD robustness evaluation",
  link=FR + "04_adversarial_attacks.ipynb", date="2026-04-02", version="adv-r1", model="det-v1.3")
a("Sn5.3", "Solution", "CAN intrusion detection", artefact="IDS detection-rate evaluation",
  link=CY + "04_can_bus_intrusion_detection.ipynb", date="2026-09-30", version="ids-r1")
aeb.supported_by("S1", "G5"); aeb.in_context_of("G5", "C5"); aeb.supported_by("G5", "S5")
aeb.in_context_of("S5", "A2")
aeb.supported_by("S5", "G5.1", "G5.2", "G5.3", "G5.4", "G5.5")
aeb.supported_by("G5.1", "Sn5.1"); aeb.supported_by("G5.2", "Sn5.2"); aeb.supported_by("G5.3", "Sn5.3")

counts = pd.Series([nd.type for nd in aeb.nodes.values()]).value_counts()
print(f"Argument '{aeb.name}': {len(aeb.nodes)} elements, {len(aeb.edges)} relationships")
print(counts.to_string())

### 7.2 Overview and branches

A full argument rarely fits on one page. We draw an overview to depth 2 and then each branch separately — the same way GSN modules and "extended views" are used in practice.

In [ ]:
_ = render(aeb, max_depth=2, title="AEB safety argument — overview (depth 2)")
print("Text summary: G1 is supported by strategy S1, which splits into four goals: "
      + "; ".join(f"{g} '{aeb.nodes[g].text}'" for g in aeb.children("S1")) + ".")

In [ ]:
_ = render(aeb, root="G2", title="Branch G2 — ISO 26262 (instantiated safety-goal pattern)")
_ = render(aeb, root="G4", title="Branch G4 — ISO/PAS 8800")

In [ ]:
_ = render(aeb, root="G5", title="Branch G5 — cybersecurity claims feeding the safety argument", highlight=("G5.4",))
print("Text summary: G5 is argued over threat scenarios (S5, assumption A2) with sub-goals G5.1-G5.3 supported by "
      "TARA, adversarial-robustness and intrusion-detection evidence; G5.4 is undeveloped (diamond); "
      "G5.5 is an away goal in the cybersecurity case module.")

## 8. Automated checks

### 8.1 Well-formedness and open items

In [ ]:
findings = validate(aeb)
display(findings)
summarise(findings, aeb.name)
undeveloped = [n for n, nd in aeb.nodes.items() if nd.undeveloped]
print("Open items (undeveloped):", ", ".join(f"{n} '{aeb.nodes[n].text}'" for n in undeveloped) or "none")

### 8.2 Evidence traceability and freshness

A safety case is a living document: evidence ages, and evidence about an ML component becomes **stale** as soon as the model is retrained. Two checks:

* **age** — evidence older than a review interval (here 12 months) must be re-confirmed;
* **configuration** — evidence produced for a different model version than the one being released does not support the claim for this release.

In [ ]:
MAX_AGE_DAYS = 365

def evidence_table(arg):
    rows = []
    for n in arg.of_type("Solution"):
        m = arg.nodes[n].meta
        d = date.fromisoformat(m["date"])
        age = (TODAY - d).days
        issues = []
        if age > MAX_AGE_DAYS:
            issues.append(f"older than {MAX_AGE_DAYS} days")
        if m.get("model") and m["model"] != DEPLOYED_MODEL:
            issues.append(f"model {m['model']} != released {DEPLOYED_MODEL}")
        rows.append({"solution": n, "supports": ", ".join(arg.parents(n)), "claim": arg.nodes[arg.parents(n)[0]].text,
                     "evidence": m["artefact"], "notebook": m["link"], "date": m["date"], "age [days]": age,
                     "version": m["version"], "model": m.get("model", "—"),
                     "status": "STALE: " + "; ".join(issues) if issues else "current"})
    return pd.DataFrame(rows)

ev = evidence_table(aeb)
lines = ["| Solution | Supports | Evidence | Course notebook | Date | Model | Status |", "|---|---|---|---|---|---|---|"]
for _, r in ev.iterrows():
    name = r["notebook"].split("/")[-1].replace(".ipynb", "")
    lines.append(f"| {r['solution']} | {r['supports']} | {r['evidence']} | [{name}]({r['notebook']}) | {r['date']} | "
                 f"{r['model']} | {'**' + r['status'] + '**' if r['status'] != 'current' else r['status']} |")
display(Markdown("\n".join(lines)))

stale = ev[ev["status"] != "current"]
print(f"{len(ev)} evidence items; {len(stale)} stale:")
for _, r in stale.iterrows():
    print(f"  {r['solution']} (supports {r['supports']}): {r['status']}")

In [ ]:
# Propagate stale evidence upwards: a goal is 'at risk' if any supporting evidence below it is stale.
def at_risk_goals(arg, stale_solutions):
    risky = set()
    def up(n):
        for p in arg.parents(n):
            if p not in risky:
                risky.add(p); up(p)
    for s in stale_solutions:
        up(s)
    return sorted(g for g in risky if arg.nodes[g].type == "Goal")

risky = at_risk_goals(aeb, stale["solution"])
_ = render(aeb, root="G4", title="Branch G4 with goals affected by stale evidence highlighted", highlight=tuple(risky))
print("Goals whose support includes stale evidence:", ", ".join(risky))

## 9. Confidence: assurance claim points and defeaters

A well-formed argument can still be **unconvincing**. Two complementary ideas address confidence:

* **Confidence arguments** (Hawkins et al., 2011) separate the *safety argument* from a *confidence argument* about it. **Assurance claim points** mark the places — an inference, a context, a piece of evidence — where confidence must be argued, e.g. *"is the simulation trustworthy enough to support G3.2?"*. GSN v3 includes assurance claim points.
* **Eliminative argumentation** (Goodenough, Weinstock & Klein, 2013) builds confidence by identifying **defeaters** — reasons for doubt — and showing that each has been eliminated. Three kinds:
  * **rebutting** defeaters attack a *claim* (*"there is a night-time scenario in which the detector misses pedestrians"*);
  * **undermining** defeaters attack *evidence* (*"the calibration was measured on a different model version"*);
  * **undercutting** defeaters attack the *inference* (*"passing the simulation campaign does not imply safety on the road"*).

  Confidence is then expressed as the fraction of identified defeaters that have been eliminated — a Baconian rather than a probabilistic measure: *n* of *m* doubts removed.

In [ ]:
defeaters = pd.DataFrame([
    ("D1", "G4.1", "rebutting", "Children are under-represented in the training data", "eliminated", "Sn4.1"),
    ("D2", "Sn4.2a", "undermining", "Calibration measured on det-v1.3, not the released det-v1.4", "open", ""),
    ("D3", "G3.2", "undercutting", "Simulation results may not transfer to real sensor data", "open", ""),
    ("D4", "Sn3.2b", "undermining", "Zero-failure bound assumes independent test kilometres", "eliminated", "Sn3.2a"),
    ("D5", "G5.2", "rebutting", "Physical adversarial patches are outside the tested perturbation norm", "open", ""),
    ("D6", "G2.1", "undercutting", "Main channel and monitor share a voltage regulator", "eliminated", "Sn2.1a"),
    ("D7", "G5.3", "undermining", "IDS evaluated only on synthetic injection traces", "eliminated", "Sn5.3 (re-run on recorded traces)"),
], columns=["id", "target", "type", "doubt", "status", "eliminated by"])
display(defeaters)

def branch_of(node):
    # walk up to the child of S1
    while True:
        ps = aeb.parents(node)
        if not ps or ps[0] == "S1":
            return node
        node = ps[0]

defeaters["branch"] = defeaters["target"].map(branch_of)
conf = defeaters.groupby("branch")["status"].agg(identified="size", eliminated=lambda s: (s == "eliminated").sum())
conf["Baconian confidence"] = conf["eliminated"].astype(str) + "|" + conf["identified"].astype(str)
display(conf)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.barh(conf.index, conf["identified"], color=OI["vermillion"], label="open")
ax.barh(conf.index, conf["eliminated"], color=OI["green"], label="eliminated")
ax.set_xlabel("number of identified defeaters [-]"); ax.set_ylabel("argument branch")
ax.set_title("Eliminative argumentation: defeaters per branch"); ax.legend(fontsize=8)
ax.xaxis.get_major_locator().set_params(integer=True)
plt.tight_layout(); plt.show()
n_open = int((defeaters["status"] == "open").sum())
print(f"{len(defeaters) - n_open} of {len(defeaters)} defeaters eliminated overall; open: "
      + "; ".join(f"{r.id} on {r.target} ({r.type})" for r in defeaters[defeaters.status == 'open'].itertuples()))

Note how the automated checks and the defeaters reinforce each other: the stale calibration evidence found in Section 8.2 is exactly the undermining defeater D2. Tooling cannot judge whether an argument is *convincing*, but it can make sure that known weaknesses are visible rather than buried.

## 10. Exercise: add a runtime-monitoring branch

UL 4600 requires safety performance indicators that tie claims to field data. Extend the argument with a fifth branch under S1:

* **G6** — *"Operational risk is monitored and the argument is kept valid in the field"*, in the context of **C6** — *"SPI: metric with a threshold that conditions a safety-case claim (UL 4600)"*;
* a strategy **S6** — *"Argue over leading and lagging safety performance indicators"*;
* two goals: **G6.1** *"The rate of OOD-triggered fallbacks stays below its threshold"* (leading) and **G6.2** *"The rate of unintended AEB activations per 10⁶ km stays below its threshold"* (lagging);
* evidence for G6.1 from the runtime-monitoring notebook ([07/01](../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb)); G6.2 marked **undeveloped** (no field data before launch).

Validate and render the new branch. How many open items does the argument now have?

<details><summary>Answer</summary>

```python
a("G6", "Goal", "Operational risk is monitored and the argument is kept valid in the field")
a("C6", "Context", "SPI: metric with a threshold that conditions a safety-case claim (UL 4600)")
a("S6", "Strategy", "Argue over leading and lagging safety performance indicators")
a("G6.1", "Goal", "The rate of OOD-triggered fallbacks stays below its threshold")
a("G6.2", "Goal", "The rate of unintended AEB activations per 10^6 km stays below its threshold", undeveloped=True)
a("Sn6.1", "Solution", "Runtime ODD/OOD monitor and SPI log", artefact="SPI dashboard specification",
  link=RT + "01_odd_and_runtime_monitoring.ipynb", date="2026-10-01", version="spi-r1", model="det-v1.4")
aeb.supported_by("S1", "G6"); aeb.in_context_of("G6", "C6"); aeb.supported_by("G6", "S6")
aeb.supported_by("S6", "G6.1", "G6.2"); aeb.supported_by("G6.1", "Sn6.1")
summarise(validate(aeb), aeb.name)
_ = render(aeb, root="G6", title="Branch G6 — runtime monitoring and SPIs")
print([n for n, nd in aeb.nodes.items() if nd.undeveloped])   # ['G5.4', 'G6.2']
```
The argument remains well-formed and now has two open items (G5.4 and G6.2). G6.2 is a lagging indicator: before launch it cannot be supported, which is why UL 4600 also expects leading indicators such as G6.1.
</details>

**Further exercises**

1. Introduce an error on purpose — connect `S4` directly to `Sn4.3` — and confirm that rule R1 catches it.
2. Add a rebutting defeater to G1 and decide what evidence would eliminate it.

<details><summary>Answers</summary>

1. `aeb.supported_by("S4", "Sn4.3"); validate(aeb)` reports an R1 error *"Strategy S4 --SupportedBy--> Solution Sn4.3 is not permitted"*. Remove the edge again with `aeb.edges.remove(("S4", "Sn4.3", "SupportedBy"))`.
2. E.g. *"field data from a comparable function shows unintended braking at motorway gantries"*. It would be eliminated by a triggering-condition analysis that includes the situation, a design change or ODD restriction, and validation evidence for it; until then the top claim carries an open rebutting defeater.
</details>

## Key takeaways

1. A **safety case** = claims + argument + evidence; the argument, which explains *why* the evidence is sufficient, is the part most often missing.
2. **GSN** (Community Standard v3, 2021) gives arguments a precise visual and structural grammar: goals, strategies, solutions, context, assumptions, justifications, away goals, and the *SupportedBy* / *InContextOf* relationships.
3. **Patterns** and **modules** make arguments reusable and manageable; the ISO 26262 branch here was generated from a pattern over safety goals.
4. **UL 4600** is safety-case-centric and ties claims to field data through **SPIs**; **ISO/TS 5083:2025** and **ISO/PAS 8800:2024** (clause 8) also expect structured safety/assurance arguments; **AMLAS** supplies argument patterns for ML components.
5. For an AI-based AEB function, one argument must integrate **ISO 26262, ISO 21448, ISO/PAS 8800 and ISO/SAE 21434** evidence — security threats against perception are safety-relevant and belong in the argument, as motivated by Patel & Jung (2026).
6. Tooling can check **well-formedness**, **open items** and **evidence freshness** automatically; **defeaters** make the remaining doubts explicit.

## Self-assessment

1. What is the difference between evidence and argument in a safety case?
2. Which *SupportedBy* connection is not permitted in GSN: Goal→Solution, Strategy→Goal or Strategy→Solution?
3. What does a hollow diamond below a goal mean, and why is it better than leaving the goal unsupported?
4. What is a safety performance indicator in UL 4600, and why are lagging indicators alone insufficient?
5. Give one example each of a rebutting, an undermining and an undercutting defeater for the claim *"Detection confidence is calibrated"*.
6. Why does retraining the pedestrian detector affect parts of the argument that were previously complete?
7. Why should cybersecurity claims appear in the *safety* argument of an AI-based perception function?

<details><summary>Answers</summary>

1. Evidence consists of facts (reports, analyses, test results); the argument explains how and why they support the claim, including the reasoning steps (strategies) and the context in which the claim holds.
2. Strategy→Solution. A strategy is a reasoning step; it must lead to sub-goals, which are then supported by solutions.
3. The element is intentionally *undeveloped* — an acknowledged open item. Marking it makes incompleteness explicit and checkable, rather than looking like an oversight.
4. A metric, supported by evidence, with a threshold that conditions a claim in the safety case. Lagging indicators (e.g. collisions) only reveal a problem after harm; leading indicators (e.g. near misses, OOD-fallback rate) can reveal that a claim is failing before mishaps occur.
5. Rebutting: *"on night-time data the ECE is 0.15"*. Undermining: *"the calibration set was drawn from the training distribution only"*. Undercutting: *"a low ECE on average does not imply calibration for the rare pedestrian class"*.
6. Evidence about ML behaviour (calibration, robustness, validation) is tied to a model version; after retraining it no longer describes the released model, so the claims it supports must be re-established.
7. Because an attack on the perception component (adversarial input, sensor spoofing, injected messages) can cause the same hazardous behaviour as a fault or an insufficiency. If the security case is kept separate, the safety argument silently assumes the absence of attacks.
</details>

## References

1. Assurance Case Working Group (2021). *Goal Structuring Notation Community Standard*, Version 3, SCSC-141C. Safety-Critical Systems Club. https://scsc.uk/gsn
2. UL Standards & Engagement (2023). *UL 4600: Standard for Safety for the Evaluation of Autonomous Products*, 3rd ed. https://ulse.org/ul-standards-engagement/autonomous-vehicle-technology/
3. ISO/TS 5083:2025. *Road vehicles — Safety for automated driving systems — Design, verification and validation*. https://www.iso.org/standard/81920.html
4. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence*. https://www.iso.org/standard/83303.html
5. ISO 26262:2018, ISO 21448:2022 and ISO/SAE 21434:2021 — see the module notebooks linked in the evidence table.
6. Hawkins, R., Paterson, C., Picardi, C., Jia, Y., Calinescu, R., Habli, I. (2021). *Guidance on the Assurance of Machine Learning in Autonomous Systems (AMLAS)*. University of York. arXiv:2102.01564. https://arxiv.org/abs/2102.01564
7. Hawkins, R., Kelly, T., Knight, J., Graydon, P. (2011). A new approach to creating clear safety arguments. In *Advances in Systems Safety* (pp. 3–23). Springer. https://doi.org/10.1007/978-0-85729-133-2_1
8. Goodenough, J. B., Weinstock, C. B., Klein, A. Z. (2013). Eliminative induction: A basis for arguing system confidence. *Proc. 35th International Conference on Software Engineering (ICSE)*, 1161–1164. https://doi.org/10.1109/ICSE.2013.6606668
9. Diemert, S., Goodenough, J., Joyce, J., Weinstock, C. (2023). Incremental assurance through eliminative argumentation. *Journal of System Safety*, 58(1), 7–15. https://doi.org/10.56094/jss.v58i1.215
10. Patel, M., Jung, R. (2026). Integrating cybersecurity into the AI safety assurance argument: A GSN pattern for AI-based perception components in highly automated driving. In *Computer Safety, Reliability, and Security. SAFECOMP 2026 Workshops*, LNCS (pp. 763–779). Springer. https://doi.org/10.1007/978-3-032-35506-5_55

---
**Navigation:** ← [03 Standards gaps](03_standards_gaps.ipynb) · [Module index](README.md) · [Course overview](../README.md) →